In [42]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [43]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str


In [44]:
model = ChatGroq(model="llama-3.3-70b-versatile")


In [45]:
def gen_joke(state: JokeState) -> JokeState:
    topic = state["topic"]
    prompt = f"Tell me a joke about {topic}."
    joke = model.invoke(prompt).content
    return {"topic": topic, "joke": joke}


In [46]:
def explain_joke(state: JokeState) -> JokeState:
    joke = state["joke"]
    prompt = f"Explain the following joke: {joke}"
    explanation = model.invoke(prompt).content
    return {"explanation": explanation}

In [47]:
graph = StateGraph(JokeState)

graph.add_node("gen_joke", gen_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "gen_joke")
graph.add_edge("gen_joke", "explain_joke")
graph.add_edge("explain_joke", END)

#These lines tell langgraph that yu've to persist the final as well as all intermediate states
checkpointer = InMemorySaver()
joke_graph = graph.compile(checkpointer=checkpointer)

In [48]:
config1 = {"configurable": {"thread_id": "1"}}

result = joke_graph.invoke({"topic": "programming"}, config=config1)
print(result)


{'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'A clever joke. The joke is a play on words. In programming, a "bug" refers to an error or a flaw in the code. However, in the context of the joke, "bug" also refers to an insect that is attracted to light.\n\nThe punchline "light attracts bugs" is a common phrase used to describe how insects, such as moths or flies, are drawn to light sources. But in this joke, it\'s also a pun on the programming term "bug". The joke is saying that programmers prefer dark mode (a color scheme with a dark background and light text) because "light attracts bugs", implying that if they use a light mode, they will attract (programming) bugs, or errors, in their code.\n\nIt\'s a lighthearted and humorous way to poke fun at the challenges of programming and the frustrations of dealing with errors in code.'}


In [49]:
config2 = {"configurable": {"thread_id": "2"}}

result2 = joke_graph.invoke({"topic": "cricket"}, config=config2)
print(result2)

{'topic': 'cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a bug in its system! (get it?)', 'explanation': 'A clever play on words. The joke is funny because it\'s a pun. Here\'s how it works:\n\n* "Bug" has a double meaning:\n  + In computing, a "bug" refers to an error or a glitch in a system.\n  + In biology, a "bug" is an informal term for an insect, like a cricket.\n* The joke sets up the expectation that the cricket went to the doctor because it\'s an insect (a "bug") and might have a medical issue.\n* But the punchline subverts this expectation by using the word "bug" in the computing sense, implying that the cricket has a technical issue, like a computer virus, in its "system" (which is a clever way to refer to the cricket\'s body).\n* The phrase "get it?" at the end is a nod to the wordplay, acknowledging that the joke relies on a clever twist on the meaning of "bug".\n\nOverall, the joke is a lighthearted and clever play on words that uses the multi

In [50]:
joke_graph.get_state(config1)

StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'A clever joke. The joke is a play on words. In programming, a "bug" refers to an error or a flaw in the code. However, in the context of the joke, "bug" also refers to an insect that is attracted to light.\n\nThe punchline "light attracts bugs" is a common phrase used to describe how insects, such as moths or flies, are drawn to light sources. But in this joke, it\'s also a pun on the programming term "bug". The joke is saying that programmers prefer dark mode (a color scheme with a dark background and light text) because "light attracts bugs", implying that if they use a light mode, they will attract (programming) bugs, or errors, in their code.\n\nIt\'s a lighthearted and humorous way to poke fun at the challenges of programming and the frustrations of dealing with errors in code.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoi

In [51]:
joke_graph.get_state(config2)

StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a bug in its system! (get it?)', 'explanation': 'A clever play on words. The joke is funny because it\'s a pun. Here\'s how it works:\n\n* "Bug" has a double meaning:\n  + In computing, a "bug" refers to an error or a glitch in a system.\n  + In biology, a "bug" is an informal term for an insect, like a cricket.\n* The joke sets up the expectation that the cricket went to the doctor because it\'s an insect (a "bug") and might have a medical issue.\n* But the punchline subverts this expectation by using the word "bug" in the computing sense, implying that the cricket has a technical issue, like a computer virus, in its "system" (which is a clever way to refer to the cricket\'s body).\n* The phrase "get it?" at the end is a nod to the wordplay, acknowledging that the joke relies on a clever twist on the meaning of "bug".\n\nOverall, the joke is a lighthearted and clever play on word

In [52]:
list(joke_graph.get_state_history(config1))

[StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'A clever joke. The joke is a play on words. In programming, a "bug" refers to an error or a flaw in the code. However, in the context of the joke, "bug" also refers to an insect that is attracted to light.\n\nThe punchline "light attracts bugs" is a common phrase used to describe how insects, such as moths or flies, are drawn to light sources. But in this joke, it\'s also a pun on the programming term "bug". The joke is saying that programmers prefer dark mode (a color scheme with a dark background and light text) because "light attracts bugs", implying that if they use a light mode, they will attract (programming) bugs, or errors, in their code.\n\nIt\'s a lighthearted and humorous way to poke fun at the challenges of programming and the frustrations of dealing with errors in code.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpo

In [53]:
# joke_graph.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f118745-60bb-6ddc-8000-752fa3236dfd"}})
joke_graph.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f118745-60bb-6ddc-8000-752fa3236dfd", "checkpoint_ns": ""}}, {"topic": "Babar Azam"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f118777-d5cb-6018-8000-ca0fa2c1830e'}}

In [57]:
joke_graph.invoke(None, config={"configurable": {"thread_id": "1", "checkpoint_id": "1f118777-d5cb-6018-8000-ca0fa2c1830e"}})

{'topic': 'Babar Azam',
 'joke': 'Why did Babar Azam bring a ladder to the cricket field?\n\nBecause he wanted to take his batting to the next level! (get it?)',
 'explanation': 'A clever joke. Here\'s how it works:\n\nThe joke is a play on words, using the phrase "take it to the next level" in a literal and figurative sense.\n\nIn cricket, "batting" refers to the act of hitting the ball with a bat. To "take your batting to the next level" is an idiomatic expression that means to improve or elevate your batting skills.\n\nHowever, the joke adds a literal twist by introducing a ladder, which is a physical object used to reach higher levels or heights. So, when Babar Azam brings a ladder to the cricket field, it\'s a humorous and unexpected interpretation of the phrase "take it to the next level". The punchline is funny because it\'s a clever and silly connection between the literal meaning of the phrase (using a ladder to reach a higher level) and the figurative meaning (improving one\'

In [58]:
list(joke_graph.get_state_history(config1))

[StateSnapshot(values={'topic': 'Babar Azam', 'joke': 'Why did Babar Azam bring a ladder to the cricket field?\n\nBecause he wanted to take his batting to the next level! (get it?)', 'explanation': 'A clever joke. Here\'s how it works:\n\nThe joke is a play on words, using the phrase "take it to the next level" in a literal and figurative sense.\n\nIn cricket, "batting" refers to the act of hitting the ball with a bat. To "take your batting to the next level" is an idiomatic expression that means to improve or elevate your batting skills.\n\nHowever, the joke adds a literal twist by introducing a ladder, which is a physical object used to reach higher levels or heights. So, when Babar Azam brings a ladder to the cricket field, it\'s a humorous and unexpected interpretation of the phrase "take it to the next level". The punchline is funny because it\'s a clever and silly connection between the literal meaning of the phrase (using a ladder to reach a higher level) and the figurative mean